# JSON-LD File Parser

`JSONLDFile` reads CIM JSON-LD files into a graph, and `cimgraph.utils.write_json_ld` writes a graph model to one. The files follow the CGMES JSON-LD draft style: a published `@context` URL, `"urn:uuid:..."` string references, and a `dcat:Dataset` header that names the units the numbers are written in. Neither needs a database or an install extra.

## Reading a JSON-LD file

Select the CIM profile, create the connection with the file name (or a list of files to load into one graph) and pass it to a graph model. A missing file raises `FileNotFoundError` and a malformed one `json.JSONDecodeError`.

In [1]:
import os
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'
import cimprofiles.cim17.cimhub_2023 as cim

from cimgraph.databases import JSONLDFile
from cimgraph.models import FeederModel

feeder = cim.Feeder(mRID='49AD8E07-3BF9-A4E2-CB8F-C3722F837B62')
network = FeederModel(container=feeder,
                      connection=JSONLDFile('../../tests/test_models/ieee13.jsonld'),
                      distributed=False)

print(f'{len(network.graph[cim.ACLineSegment])} lines, {len(network.graph[cim.EnergyConsumer])} loads')

11 lines, 16 loads


Unit values are `CIMUnit`s stored in base SI units. Convert them with `.to()`:

In [2]:
line = network.first(cim.ACLineSegment)
load = network.first(cim.EnergyConsumer)
print(f'Line {line.name}: {line.length.to("ft"):.0f} ft')
print(f'Load {load.name}: {load.p.to("kW"):.0f} kW, {load.q.to("kVAr"):.0f} kVAr')

Line 645646: 300 ft
Load 646: 230 kW, 132 kVAr


## Model headers

Each file's `dcat:Dataset` header is read into `JSONLDFile.model_headers`, one `ModelHeader` per file. `ieee13.jsonld` is an older file without a header, so its fields are empty and its numbers are read in the profile's default units (base SI for `cimhub_2023`).

In [3]:
network.connection.model_headers

[ModelHeader(full_model_uuid=None, created=None, description=None, model_name=None, authority=None, authority_uuid=None)]

## Writing a JSON-LD file

`write_json_ld(network, filename, indent=4, convention=None, header=None, contexts=None)` writes the model with the header as the top-level map and the objects in its `@graph`.

* `convention` names the units convention of the numbers. `cimhub_2023` has `'si'` (the default: W, V, m) and `'distribution'` (kW, kVAr, kVA, line constants per km, wire sizes in mm). The header names the convention's units vocabularies in `dct:conformsTo`.
* `header` adds or replaces header properties. A fixed `@id`, `dct:identifier` and `dct:issued` make repeated exports of the same model identical files; by default they are a new UUID and the export time.
* `contexts` are further JSON-LD contexts, for header prefixes that the profile's context does not define.

In [4]:
import json
import tempfile
from pathlib import Path

from cimgraph.utils import write_json_ld

output_dir = Path(tempfile.mkdtemp())
path = output_dir / 'ieee13.jsonld'
write_json_ld(network, str(path), convention='distribution', header={
    '@id': 'urn:uuid:5a0c3a4e-0000-5000-8000-000000000001',
    'dct:identifier': '5a0c3a4e-0000-5000-8000-000000000001',
    'dct:issued': '2026-11-01T00:00:00Z',
    'dcat:isVersionOf': 'https://example.org/models/ieee13',
})

document = json.loads(path.read_text())
header = {key: value for key, value in document.items() if key != '@graph'}
print(json.dumps(header, indent=2))

{
  "@context": "https://pnnl-cim-tools.github.io/CIM-Profiles/cimhub_2023/0.2.0/context.jsonld",
  "@id": "urn:uuid:5a0c3a4e-0000-5000-8000-000000000001",
  "@type": [
    "dcat:Dataset",
    "http://www.w3.org/2004/03/trix/rdfg-1/Graph"
  ],
  "dct:identifier": "5a0c3a4e-0000-5000-8000-000000000001",
  "dct:issued": "2026-11-01T00:00:00Z",
  "md:Model.modelingAuthoritySet": "https://github.com/PNNL-CIM-Tools/CIM-Graph",
  "dct:conformsTo": [
    "https://pnnl-cim-tools.github.io/CIM-Profiles/cimhub_2023/0.2.0/units-distribution.jsonld"
  ],
  "dcat:distribution": {
    "@type": "dcat:Distribution",
    "dct:conformsTo": [
      "https://pnnl-cim-tools.github.io/CIM-Profiles/cimhub_2023/0.2.0/units-distribution.jsonld"
    ]
  },
  "dcat:isVersionOf": "https://example.org/models/ieee13"
}


The load is written in kW, as the `distribution` convention says:

In [5]:
written_load = next(node for node in document['@graph'] if node['@id'] == f'urn:uuid:{load.uri()}')
print(json.dumps(written_load, indent=2))

{
  "@id": "urn:uuid:0BC9183A-9067-4E06-AA5B-48DC2AB30C80",
  "@type": "cim:EnergyConsumer",
  "cim:EnergyConsumer.customerCount": "1",
  "cim:EnergyConsumer.grounded": "false",
  "cim:EnergyConsumer.p": "230",
  "cim:EnergyConsumer.phaseConnection": "cim:PhaseShuntConnectionKind.D",
  "cim:EnergyConsumer.q": "132",
  "cim:EnergyConsumer.LoadResponse": "urn:uuid:E84837F3-04A7-414B-A72C-B429DF35BBBF",
  "cim:ConductingEquipment.BaseVoltage": "urn:uuid:2A158E0C-CD01-4A50-AEBA-59D761FCF15D",
  "cim:Equipment.EquipmentContainer": "urn:uuid:49AD8E07-3BF9-A4E2-CB8F-C3722F837B62",
  "cim:PowerSystemResource.Location": "urn:uuid:9E735173-F2E5-408C-8279-54ED0D3BBDD1",
  "cim:IdentifiedObject.mRID": "0BC9183A-9067-4E06-AA5B-48DC2AB30C80",
  "cim:IdentifiedObject.name": "646"
}


Reading the file back gives the header and the same values in base SI units:

In [6]:
reader = JSONLDFile(str(path))
print(reader.model_headers)

copy = FeederModel(container=feeder, connection=reader, distributed=False)
print(f'{copy.graph[cim.EnergyConsumer][load.identifier].p.to("kW"):.0f} kW')

[ModelHeader(full_model_uuid='5a0c3a4e-0000-5000-8000-000000000001', created='2026-11-01T00:00:00Z', description=None, model_name=None, authority='https://github.com/PNNL-CIM-Tools/CIM-Graph', authority_uuid='bdf23f0c-0c08-525b-846a-e05b60f8af02')]
230 kW


## CGMES-style files

`JSONLDFile` also reads files in the Nordic44 layout, where the header is a node in `@graph` and its `dct:conformsTo` names the CGMES vocabularies (kV, MW, km). `cgmes_style_eq.jsonld` is a small CGMES 3.0 Equipment file of this kind. Its `@context` is a URL, which is resolved from the cimprofiles offline cache.

In [7]:
os.environ['CIMG_CIM_PROFILE'] = 'cgmes_3_0_0'
os.environ['CIMG_AUTHORITY_FROM_IS_VERSION_OF'] = 'true'   # its authority is in dcat:isVersionOf
import cimprofiles.cim17.cgmes_3_0_0 as cgmes

from cimgraph.models import NodeBreakerModel

reader = JSONLDFile('../../tests/test_models/cgmes_style_eq.jsonld')
print(reader.model_headers)
print('nominalVoltage is written in', reader.units['BaseVoltage.nominalVoltage'])

[ModelHeader(full_model_uuid='7b1e0c2a-1d3f-4b6e-9a51-000000000001', created=None, description=None, model_name=None, authority='https://github.com/PNNL-CIM-Tools/CIM-Graph/tests', authority_uuid='bdb0ee55-1d03-5def-af93-c8e742aa1619')]
nominalVoltage is written in kV


In [8]:
eq_model = NodeBreakerModel(container=None, connection=reader, distributed=False)

line = eq_model.first(cgmes.ACLineSegment)
machine = eq_model.first(cgmes.SynchronousMachine)
print(f'{line.name}: {line.length.to("km")} km, {line.BaseVoltage.nominalVoltage.to("kV")} kV')
print(f'{machine.name}: {machine.ratedS.to("MVA")} MVA')

420NORTH-SOUTH 1: 12.5 km, 420.0 kV
NORTH G1: 680.0 MVA


The header's model authority (`md:Model.modelingAuthoritySet`) sets the model scope used to seed UUIDs, as `md:FullModel` does for `XMLFile`. Its `dct:title` is the file name (`description`), so the model name stays as it is. `dcat:isVersionOf` names a model series, so it is the authority only with `CIMG_AUTHORITY_FROM_IS_VERSION_OF=true`, as set above for this CGMES file. Without it, a header that has no `md:Model.modelingAuthoritySet` leaves the model scope as it is, with a warning:

In [9]:
from cimgraph.core import get_model_authority

get_model_authority()

'https://github.com/PNNL-CIM-Tools/CIM-Graph/tests'

## Remote contexts

Contexts and unit vocabularies that are not in the cimprofiles offline cache are fetched only with `CIMG_ALLOW_REMOTE_CONTEXT=true`. Otherwise reading such a file fails with an error that names the URL.

Files loaded together into one `JSONLDFile` must use the same units; files that write an attribute in different units raise `ValueError`.

See also: [XML File Parser](3_8_xml_file_parser.ipynb), [Environment Variables](3_2_env_variables.ipynb).